<a href="https://colab.research.google.com/github/SRET-College/Sem-6-NN-and-DL/blob/main/NN_and_DL_Expt_4b_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

Wed Mar  4 17:35:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.optimizers import Adam

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
(X_train, y_train), (X_test, y_test)= fashion_mnist.load_data()


# Normalize pixel values from [0,255] to [0,1]
# Normalization improves convergence speed and stability
X_train = X_train / 255.0
X_test = X_test / 255.0

In [ ]:
# Learning rates to be tested
learning_rates = [0.01, 0.001]

# Number of neurons in hidden layer
hidden_units = [128, 256]

# Batch sizes for training
batch_sizes = [32, 64]

In [ ]:
best_accuracy = 0
best_params = {}

with tf.device('/GPU:0'):

    for lr in learning_rates:

        for units in hidden_units:

            for batch in batch_sizes:

                model = Sequential([
                    tf.keras.Input(shape=(28, 28)),
                    Flatten(),
                    Dense(units, activation='relu'),
                    Dense(10, activation='softmax')
                ])

                model.compile(
                    optimizer=Adam(learning_rate=lr),
                    loss='sparse_categorical_crossentropy',
                    metrics=['accuracy']
                )

                model.fit(
                    X_train, y_train,
                    epochs=30,
                    batch_size=batch,
                    validation_split=0.2,
                    verbose=0
                )

                _, acc = model.evaluate(X_test, y_test, verbose=0)

                print(f"LR={lr}, Units={units}, Batch={batch} → Accuracy={acc:.4f}")

                if acc > best_accuracy:
                    best_accuracy = acc
                    best_params = {
                        'learning_rate': lr,
                        'units': units,
                        'batch_size': batch
                    }

LR=0.01, Units=128, Batch=32 → Accuracy=0.8447
LR=0.01, Units=128, Batch=64 → Accuracy=0.8550
LR=0.01, Units=256, Batch=32 → Accuracy=0.8485
LR=0.01, Units=256, Batch=64 → Accuracy=0.8645
LR=0.001, Units=128, Batch=32 → Accuracy=0.8876
LR=0.001, Units=128, Batch=64 → Accuracy=0.8862
LR=0.001, Units=256, Batch=32 → Accuracy=0.8894
LR=0.001, Units=256, Batch=64 → Accuracy=0.8906


In [ ]:
print(f"\nBest Accuracy from Grid Search: {best_accuracy * 100:.2f}")
print(f"Best Hyperparameters: {best_params}")


Best Accuracy from Grid Search: 88.90
Best Hyperparameters: {'learning_rate': 0.001, 'units': 256, 'batch_size': 64}


In [ ]:
import random

# 1. Define your parameter space
learning_rates = [0.01, 0.001, 0.0001]
hidden_units = [128, 256, 512]
batch_sizes = [32, 64, 128]

num_iterations = 50
results = []

# 2. Random Search Loop
for i in range(num_iterations):
    # Pick random values
    lr = random.choice(learning_rates)
    units = random.choice(hidden_units)
    batch = random.choice(batch_sizes)

    print(f"Iteration {i+1}/{num_iterations}: Testing LR={lr}, Units={units}, Batch={batch}")

    model = Sequential(
        [
            keras.Input(shape=(28, 28)), # Corrected input shape for Fashion MNIST
            Flatten(),
            Dense(units, activation='relu'),
            Dense(10, activation='softmax'),
        ]
    )

    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )

    # Train for a few epochs to see potential
    history = model.fit(
        X_train,
        y_train,
        epochs=30,
        batch_size=batch,
        validation_split=0.2,
        verbose=0,
    )

    val_acc = max(history.history['val_accuracy'])
    results.append(
        {
            'params': {'learning_rate': lr, 'units': units, 'batch_size': batch},
            'accuracy': val_acc,
        }
    )

# 3. Identify the winner
best_run = max(results, key=lambda x: x['accuracy'])
best_params = best_run['params']

print(f"\nRandom Search Complete!")
print(f"Best Accuracy Found: {best_run['accuracy']:.4f}")
print(f"Best Parameters: {best_params}")

Iteration 1/50: Testing LR=0.0001, Units=256, Batch=32
Iteration 2/50: Testing LR=0.001, Units=256, Batch=32
Iteration 3/50: Testing LR=0.01, Units=512, Batch=128
Iteration 4/50: Testing LR=0.01, Units=128, Batch=64
Iteration 5/50: Testing LR=0.001, Units=512, Batch=128
Iteration 6/50: Testing LR=0.01, Units=256, Batch=128
Iteration 7/50: Testing LR=0.01, Units=256, Batch=128
Iteration 8/50: Testing LR=0.001, Units=256, Batch=64
Iteration 9/50: Testing LR=0.01, Units=256, Batch=128
Iteration 10/50: Testing LR=0.01, Units=512, Batch=64
Iteration 11/50: Testing LR=0.01, Units=256, Batch=128
Iteration 12/50: Testing LR=0.01, Units=512, Batch=32
Iteration 13/50: Testing LR=0.0001, Units=512, Batch=64
Iteration 14/50: Testing LR=0.001, Units=128, Batch=64
Iteration 15/50: Testing LR=0.01, Units=512, Batch=128
Iteration 16/50: Testing LR=0.001, Units=512, Batch=32
Iteration 17/50: Testing LR=0.0001, Units=128, Batch=128
Iteration 18/50: Testing LR=0.01, Units=512, Batch=32
Iteration 19/50: T